# Static SHRED on ME4OH full-field OFAM3 SST data - changing lag length

Note: Assumptions as of 30/06/26
- OFAM3 data is stored for Thursdays across the 1979-2014 time period.
- Pending confirmation from the ME4OH team, I have assumed that the data is the full-field SST values generated by the simulator model (OFAM3) averaged across a calendar week

In [ ]:
# Standard imports
import json
from collections import Counter

# Third-party imports
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import ListedColormap
from skimage.metrics import structural_similarity as SSI
from sklearn.metrics import mean_absolute_error as MAE
from sklearn.metrics import mean_squared_error as MSE
from sklearn.metrics import r2_score

# Local imports
from Data.preprocess_sst_ofam3_shred import NA_DATA_PATH
from helper_functions.evaluation import (
    area_weighted_mean,
    get_lats_longs,
    mask_array_by_lat_long,
)
from helper_functions.plotting_functions import plot_compare_sst_recon
from helper_functions.saving import unpack_metadata

np.random.seed(42)

In [ ]:
colours = [ "#ffb000", "#dc267f", "#1589e8", "#631ff3", "#fe5100", "#000000"]  # IBM colourblind palette

## SHRED preprocessing

In [ ]:
# Load and reformat data
lats, longs = get_lats_longs(NA_DATA_PATH)

In [ ]:
num_sensors = 5
lags = 2
test_recons_5_2 = np.load(f"Reconstructions/Static/Exp3/recons_n{num_sensors}_l{lags}.npy")
ground_truth_5_2 = np.load(f"Reconstructions/Static/Exp3/truth_n{num_sensors}_l{lags}.npy")
sensor_locs_5_2, sensor_coords_5_2, test_dates_5_2 = unpack_metadata(f"Reconstructions/Static/Exp3/metadata_n{num_sensors}_l{lags}.json")

num_sensors = 5
lags = 52
test_recons_5_52 = np.load(f"Reconstructions/Static/Exp1/recons_n{num_sensors}_l{lags}.npy")
ground_truth_5_52 = np.load(f"Reconstructions/Static/Exp1/truth_n{num_sensors}_l{lags}.npy")
sensor_locs_5_52, sensor_coords_5_52, test_dates_5_52 = unpack_metadata(f"Reconstructions/Static/Exp1/metadata_n{num_sensors}_l{lags}.json")

In [ ]:
# Update with models with larger internal dimensions: experiment 3.1
# To use models with internal dimension 64 -> 1059, 2 data is in exp3; 1059, 52 data is in exp2
num_sensors = 1059
lags = 2
test_recons_1059_2 = np.load(f"Reconstructions/Static/Exp3/recons_n{num_sensors}_l{lags}.npy")
ground_truth_1059_2 = np.load(f"Reconstructions/Static/Exp3/truth_n{num_sensors}_l{lags}.npy")
sensor_locs_1059_2, sensor_coords_1059_2, test_dates_1059_2 = unpack_metadata(f"Reconstructions/Static/Exp3-1/metadata_n{num_sensors}_l{lags}.json")

num_sensors = 1059
lags = 52
test_recons_1059_52 = np.load(f"Reconstructions/Static/Exp2/recons_n{num_sensors}_l{lags}.npy")
ground_truth_1059_52 = np.load(f"Reconstructions/Static/Exp2/truth_n{num_sensors}_l{lags}.npy")
sensor_locs_1059_52, sensor_coords_1059_52, test_dates_1059_52 = unpack_metadata(f"Reconstructions/Static/Exp3-1/metadata_n{num_sensors}_l{lags}.json")

## Evaluation

### n=5, l=2

In [ ]:
for i in np.random.choice(range(len(test_recons_5_2)), 1):
    recon_df = pd.DataFrame({
        "Latitude": lats,
        "Longitude": longs,
        "SST": test_recons_5_2[i]
    })
    true_df = pd.read_csv(f"{NA_DATA_PATH}{test_dates_5_2[i]}.csv")
    print(f"Date: {test_dates_5_2[i]}")
    # plot_compare_sst_recon(recon_df, true_df, area="NA")
    plot_compare_sst_recon(recon_df, true_df, area="NA", sensor_coords=sensor_coords_5_2)

### n=1059, l=2

In [ ]:
for i in np.random.choice(range(len(test_recons_1059_2)), 1):
    recon_df = pd.DataFrame({
        "Latitude": lats,
        "Longitude": longs,
        "SST": test_recons_1059_2[i]
    })
    true_df = pd.read_csv(f"{NA_DATA_PATH}{test_dates_1059_2[i]}.csv")
    print(f"Date: {test_dates_1059_2[i]}")
    # plot_compare_sst_recon(recon_df, true_df, area="NA")
    plot_compare_sst_recon(recon_df, true_df, area="NA", sensor_coords=sensor_coords_1059_2)

### Metrics
Structural Similarity Index: [[1]](https://scikit-image.org/docs/stable/api/skimage.metrics.html#skimage.metrics.structural_similarity) [[2]](https://ieeexplore.ieee.org/document/4775883)



#### NA basin

In [ ]:
for (num_sensors, test, truth) in [(5, test_recons_5_2, ground_truth_5_2), (1059, test_recons_1059_2, ground_truth_1059_2)]:
    print("NUM SENSORS: ", num_sensors, " LAGS: 2")
    rmse = np.sqrt(MSE(y_pred=test, y_true=truth))
    mae = MAE(y_pred=test, y_true=truth)
    r2 = r2_score(y_pred=test, y_true=truth)
    print(f"RMSE:\t{rmse:.3f}")
    print(f"MAE:\t{mae:.3f}")
    print(f"R2:\t{r2:.3f}")

    ssi = np.mean([SSI(truth[i], test[i], 
                   data_range=(max(test[i].max(), truth[i].max()) - min(test[i].min(), truth[i].min())))
                   for i in range(len(test))])
print(f"Average SSI:\t{ssi:.3f}")

#### Gulf-Stream region

In [ ]:
for (num_sensors, test, truth) in [(5, test_recons_5_2, ground_truth_5_2), (1059, test_recons_1059_2, ground_truth_1059_2)]:
    print("NUM SENSORS: ", num_sensors, " LAGS: 2")
    GS_test_data = mask_array_by_lat_long(test, lats, longs)
    GS_truth_data = mask_array_by_lat_long(truth, lats, longs)
    GS_test_data = [d[~np.isnan(d)] for d in GS_test_data]
    GS_truth_data = [d[~np.isnan(d)] for d in GS_truth_data]

    rmse = np.sqrt(MSE(y_pred=GS_test_data, y_true=GS_truth_data))
    mae = MAE(y_pred=GS_test_data, y_true=GS_truth_data)
    r2 = r2_score(y_pred=GS_test_data, y_true=GS_truth_data)
    print(f"RMSE:\t{rmse:.3f}")
    print(f"MAE:\t{mae:.3f}")
    print(f"R2:\t{r2:.3f}")

    ssi = np.mean([SSI(GS_truth_data[i], GS_test_data[i], data_range=test[i].max() -  test[i].min()) for i in range(len(test))])
    print(f"Average SSI:\t{ssi:.3f}")

### Regional integral of SST (area weighted mean)

In [ ]:
for (lags, test_5, test_1059, truth, dates) in [(2, test_recons_5_2, test_recons_1059_2, ground_truth_5_2, test_dates_5_2),
                                                (52, test_recons_5_52, test_recons_1059_52, ground_truth_5_52, test_dates_5_52)]:
    
    test_sst_mean_5, _ = area_weighted_mean(test_5, lats, longs)
    test_sst_mean_1059, _ = area_weighted_mean(test_1059, lats, longs)
    truth_sst_mean, _ = area_weighted_mean(truth, lats, longs)
    print()
    sst_mean_df = pd.DataFrame({
        "Date": dates,
        "SHRED - 5 sensors": test_sst_mean_5,
        "SHRED - 1059 sensors": test_sst_mean_1059,
        "Truth": truth_sst_mean
    })
    # Smooth over 1 year = 52 weeks
    sst_rolling_df = pd.DataFrame({
        "Date": dates,
        "SHRED - 5 sensors": sst_mean_df["SHRED - 5 sensors"].rolling(52).mean(),
        "SHRED - 1059 sensors": sst_mean_df["SHRED - 1059 sensors"].rolling(52).mean(),
        "Truth": sst_mean_df.Truth.rolling(52).mean()
    })
    sst_mean_df = sst_mean_df.set_index("Date")
    sst_rolling_df = sst_rolling_df.set_index("Date")

    sst_mean_df[["SHRED - 5 sensors", "SHRED - 1059 sensors", "Truth"]].plot(colormap=ListedColormap(colours[:3]), ls='-.')
    plt.xticks(rotation=90)
    plt.ylabel("Mean SST ($\\degree$C)")
    plt.title(f"{lags} week sequences")
    plt.legend(bbox_to_anchor=(1,1))
    plt.show()

    plt.figure(figsize=(8,8))
    sst_rolling_df[["SHRED - 5 sensors", "SHRED - 1059 sensors", "Truth"]].plot(colormap=ListedColormap(colours[:3]), ls='-.')
    plt.xticks(rotation=90)
    plt.ylabel("Smoothed mean SST ($\\degree$C)")
    plt.legend(bbox_to_anchor=(1,1))
    plt.title(f"{lags} week sequences")
    plt.show()

#### Zonally integrated average difference 
i.e. Average difference between reconstruction and truth across all years;
plotted against latitude (no weighting by area needed, because each square in latitude band has same area)

In [ ]:
years = [int(date.year) for date in test_dates_5_2]
year_labels = np.unique(years)
Counter(years)

In [ ]:
years = [int(date.year) for date in test_dates_5_52]
year_labels = np.unique(years)
Counter(years)

In [ ]:
avg_all_time_lat_avg_diffs = []
labels = []
for (num_sensors, lags, test, truth) in [(5, 2, test_recons_5_2, ground_truth_5_2),
                      (5, 52, test_recons_5_52, ground_truth_5_52),
                      (1059, 2, test_recons_1059_2, ground_truth_1059_2),
                      (1059, 52, test_recons_1059_52, ground_truth_1059_52)]:
    
    # For each reconstruction at a set date, find the average difference at each latitude
    difference = test - truth
    latitude_avg_diffs = np.array([pd.Series(diff_at_time).groupby(lats).mean().to_numpy() for diff_at_time in difference])

    # For each latitude, calculate the mean difference over all time
    avg_all_time_lat_avg_diffs.append(np.array([np.mean(latitude_avg_diffs.T[j]) for j in range(latitude_avg_diffs.shape[1])]))
    labels.append(f"{num_sensors} sensors, {lags} week sequences")


In [ ]:
# Switch colours so that can compare to graph in exp 1/2 directly :)
colours = ["#1589e8", "#ffb000", "#631ff3", "#dc267f"]

In [ ]:

for i in range(len(avg_all_time_lat_avg_diffs)):
    plt.plot(avg_all_time_lat_avg_diffs[i], np.unique(lats), c=colours[i], label=labels[i])
    plt.vlines(0, 0, 60, colors=['k'], linestyles=['--'])
    plt.xlabel('Average difference (from the truth) ($\\degree$C)')
    plt.ylabel('Latitude')
    plt.legend(title="Set up", bbox_to_anchor=(1,1))

Thoughts:
- Why is 1059, 2 week sequences looking better than 1059, 5 week in the average difference by latitude plot?
    - Well, better at the GS range of latitudes that is
    - This DOESN'T show in the GS area metrics